# Python Protocols, Iterators, Generators and Context Managers
**Software Development & Python — practical seminar**

One evolving class, `DataStream`, connects familiar syntax to behavior.
Work in pairs: **predict → run → inspect → explain**. For every experiment, ask:
**What protocol is being used here?**

Optional generator-state exploration and SupportsReset replace discussion time or
become homework. The integration task consolidates code written earlier.


## How to use this notebook
Python 3.10+; all lab code uses the standard library. Python 3.15 features are
checked before use. No installation is needed for the examples.

Run from the top. Each new `class DataStream` replaces the previous definition;
create fresh instances after each replacement. Earlier instances keep their old class.
The manual stage accepts an indexable sequence; the generator stage accepts any iterable.

Exercise cells contain `NotImplementedError`. Their check cells report **TODO**
until implemented; assertion failures remain failures. Later demonstrations use a
fresh class so an unfinished exercise does not block the lab. Complete solutions
are only at the very end. The supplied outputs were generated before students edit TODOs.


In [1]:
import builtins
import sys
from collections.abc import Iterable, Iterator, Mapping
from itertools import islice

print("Python:", sys.version.split()[0])
print("frozendict available:", hasattr(builtins, "frozendict"))
print("sentinel available:", hasattr(builtins, "sentinel"))


Python: 3.12.3
frozendict available: False
sentinel available: False


## 1. Warm-up: What Is a Protocol?
**Predict:** How does Python know whether `len(x)`, `for item in x`, `with x`,
or `x()` makes sense? Name the behavior each operation requests.

```text
syntax → protocol → special methods on the type → object behavior
```

| Syntax | Behavioral interface |
|---|---|
| `len(x)` | `__len__` |
| `for item in x` | `iter(x)`, then repeated `next(...)` |
| `with x` | `__enter__`, `__exit__` |
| `x()` | `__call__` |

Python often cares about behavior rather than concrete class identity.


In [2]:
class DataStream:
    def __init__(self, data):
        self.data = data

stream = DataStream([10, 20, 30])
try:
    print(list(stream))
except TypeError as exc:
    print(type(exc).__name__ + ":", exc)


TypeError: 'DataStream' object is not iterable


**Inspect → explain:** Having an attribute containing a list does not make the
containing object iterable. Which method must `DataStream` provide?
Keep that question open while we inspect how Python recognizes an interface.


## 2. Why isinstance(x, Iterable) Can Be True
**Predict all three results before running.** There is no declared inheritance here.


In [ ]:
class MyCollection:
    def __iter__(self):
        return iter([1, 2, 3])

x = MyCollection()
print("instance:", isinstance(x, Iterable))
print("subclass:", issubclass(MyCollection, Iterable))
print("Iterable in MRO:", Iterable in MyCollection.__mro__)
print("MRO:", MyCollection.__mro__)


instance: True
subclass: True
Iterable in MRO: False
MRO: (<class '__main__.MyCollection'>, <class 'object'>)


**Explain:** `Iterable` uses abstract base class (ABC) machinery. Its metaclass,
`abc.ABCMeta`, supplies the instance/subclass checks. For this ordinary instance,
we can understand the successful check along this conceptual path:

```text
isinstance(x, Iterable)
    ↓ subclass check for type(x)
ABC machinery (including caches and registered subclasses)
    ↓ Iterable.__subclasshook__(MyCollection)
recognizes a non-None __iter__ in the class hierarchy
    ↓ True
virtual subclass: recognized without inheritance in the MRO
```

This is a conceptual path, not a literal trace of every cached call.
The hook is specific to this ABC: **not all ABCs inspect arbitrary methods**.


In [4]:
from abc import ABC, ABCMeta, abstractmethod

print("hook:", Iterable.__subclasshook__)
print("metaclass is ABCMeta:", isinstance(Iterable, ABCMeta))
print("hook answer:", Iterable.__subclasshook__(MyCollection))
assert Iterable.__subclasshook__(MyCollection) is True


hook: <bound method Iterable.__subclasshook__ of <class 'collections.abc.Iterable'>>
metaclass is ABCMeta: True
hook answer: True


**Predict:** Does recognition prove that iteration works correctly?


In [5]:
class BrokenIterable:
    def __iter__(self):
        return 42  # Not an iterator!

print("recognized:", isinstance(BrokenIterable(), Iterable))
try:
    iter(BrokenIterable())
except TypeError as exc:
    print("iteration fails:", exc)


recognized: True
iteration fails: iter() returned non-iterator of type 'int'


### Edge case: iteration without Iterable recognition
`iter(obj)` can fall back to `__getitem__(0)`, `__getitem__(1)`, … until `IndexError`.
The `Iterable` hook does not check this legacy sequence route.
**Predict:** Will `list(...)` and `isinstance(...)` agree here?


In [11]:
class OldSequence:
    def __getitem__(self, index):
        if index >= 3:
            raise IndexError
        return index * 10

legacy = OldSequence()
print("values:", list(legacy))
print("recognized:", isinstance(legacy, Iterable))
assert list(legacy) == [0, 10, 20]
assert not isinstance(legacy, Iterable)


values: [0, 10, 20]
recognized: False


**Explain:** “can be iterated” and “recognized by the Iterable ABC”
are different questions. To actually start iteration, use `iter(obj)` and handle
`TypeError` where appropriate. Even successful `iter()` does not prove that every
future `next()` will succeed.


## 3. Nominal vs Structural Compatibility
A **nominal** relationship is declared by inheritance. A **structural** relationship
comes from providing the required interface. Virtual subclass recognition does not
insert a base into the MRO or inject its methods.


In [7]:
class ExplicitCollection(Iterable):
    def __iter__(self):
        return iter([1, 2, 3])

for cls in (ExplicitCollection, MyCollection):
    print(cls.__name__, "subclass:", issubclass(cls, Iterable),
          "nominal:", Iterable in cls.__mro__)


ExplicitCollection subclass: True nominal: True
MyCollection subclass: True nominal: False


### Hook recognition and explicit registration
Registration is another way an ABC can recognize a virtual subclass. It is a
promise made by the programmer, not a method implementation check.
**Predict:** Will a plain ABC recognize a matching method automatically?


In [24]:
class CloseABC(ABC):
    @abstractmethod
    def close(self):
        pass

class FileLike:
    def close(self) -> None:
        print("closed")

print("before registration:", issubclass(FileLike, CloseABC))
assert not issubclass(FileLike, CloseABC)
CloseABC.register(FileLike)
print("after registration:", issubclass(FileLike, CloseABC))
print("ABC in MRO:", CloseABC in FileLike.__mro__)
assert CloseABC not in FileLike.__mro__


before registration: False
after registration: True
ABC in MRO: False


`Iterable` supplied a structural hook; `CloseABC` did not. Here registration
explicitly declares virtual compatibility. Neither mechanism is `typing.Protocol`.


## 4. typing.Protocol
**Predict:** Can a function ask for “anything with `close() -> None`” without
requiring `FileLike` to inherit a base class?


In [25]:
from typing import Protocol, runtime_checkable

class SupportsClose(Protocol):
    def close(self) -> None:
        ...

def finish(resource: SupportsClose) -> None:
    resource.close()

finish(FileLike())  # Statically compatible; no SupportsClose base required.
try:
    isinstance(FileLike(), SupportsClose)
except TypeError as exc:
    print("undecorated Protocol:", exc)


closed
undecorated Protocol: Instance and class checks can only be used with @runtime_checkable protocols


`Protocol` primarily describes **static structural typing**: a type checker compares
members and signatures. Python does not enforce function annotations when calling
`finish`. No static checker is required or run in this notebook.
For an explicitly requested runtime check, add `@runtime_checkable`.


In [28]:
@runtime_checkable
class SupportsClose(Protocol):
    def close(self) -> None:
        ...


class FileLike:
    def close(self) -> None:
        print("closed")

print(isinstance(FileLike(), SupportsClose))
print(SupportsClose in FileLike.__mro__)


True
False


In [11]:
class WrongSignature:
    def close(self, required_argument):
        return "not the promised result"

print("runtime check:", isinstance(WrongSignature(), SupportsClose))
try:
    finish(WrongSignature())  # A static checker should reject this call.
except TypeError as exc:
    print("actual call:", exc)


runtime check: True
actual call: WrongSignature.close() missing 1 required positional argument: 'required_argument'


**Inspect → explain:** Runtime Protocol checks are shallow: member presence does
not validate full signatures, return types, or semantics.

| Aspect | `collections.abc.Iterable` | `typing.Protocol` |
|---|---|---|
| Main role | Standard runtime behavioral interface; also usable in annotations | Static structural typing, including custom interfaces |
| Mechanism | ABCMeta, subclass logic, registration; Iterable has a hook | Type checker compares the declared members |
| Inheritance required? | No: virtual subclasses are possible | No: structural compatibility |
| Runtime `isinstance` / `issubclass` | Supported | Requires `@runtime_checkable`; `issubclass` has further restrictions for data members |
| Guarantees | Recognition, not correct implementation | Static signatures when checked; runtime presence checks are limited |

Related ideas, distinct mechanisms. What did each of our three checks actually prove?


## 5. DataStream: Iterable vs Iterator
Return to our class: `list(stream)` failed. It needs to produce an iterator.
An **iterable** can supply a traversal; an **iterator** holds the traversal state.
**Predict the four Boolean values.**


In [40]:
numbers = [1, 2, 3]
it = iter(numbers)
print("list:", isinstance(numbers, Iterable), isinstance(numbers, Iterator))
print("iterator:", isinstance(it, Iterable), isinstance(it, Iterator))
print("iterator returns itself:", iter(it) is it)
print(next(it), next(it), next(it))
try:
    next(it)
except StopIteration:
    print("exhausted: StopIteration")
assert isinstance(it, Iterable) and isinstance(it, Iterator)
assert not isinstance(numbers, Iterator)


list: True False
iterator: True True
iterator returns itself: True
1 2 3
exhausted: StopIteration


| Role | Required methods | Typical `iter(x)` result |
|---|---|---|
| Iterable | `__iter__` (modern route) | An iterator |
| Iterator | `__iter__`, `__next__` | Itself |

Every conforming iterator is iterable; not every iterable is an iterator.
A `for` loop catches `StopIteration` as normal completion. Other exceptions propagate.
An exhausted iterator must remain exhausted.


## 6. Implementing an Iterator Manually
### Exercise 1 — Manual Iterator
Implement only `DataStreamIterator.__next__`. Return one item, advance the index,
and raise `StopIteration` after the last item. Keep raising it on later calls.
This stage assumes an indexable sequence that is not mutated during traversal.

**Predict:** Why should the stream create a fresh iterator instead of storing one
shared index on itself? Think about two interleaved traversals.


In [ ]:
class DataStreamIterator:
    def __init__(self, data):
        self.data = data
        self.index = 0

    def __iter__(self):
        return self

    def __next__(self):
        # TODO: implement traversal and exhaustion.
        raise NotImplementedError("Exercise 1: __next__")

    

class DataStream:
    def __init__(self, data):
        self.data = data

    def __iter__(self):
        return DataStreamIterator(self.data)

ManualDataStream = DataStream  # Keep this stage for its later solution tests.


In [43]:
def check_manual(Stream):
    assert list(Stream([1, 2, 3])) == [1, 2, 3]
    assert list(Stream([])) == []
    stream = Stream([10, 20, 30])
    assert iter(stream) is not stream
    a, b = iter(stream), iter(stream)
    assert iter(a) is a
    assert isinstance(a, Iterator)
    assert next(a) == 10 and next(a) == 20
    assert next(b) == 10  # Independent traversal state.
    assert next(a) == 30
    for _ in range(2):
        try:
            next(a)
        except StopIteration:
            pass
        else:
            raise AssertionError("An exhausted iterator must stay exhausted")
    print("Manual iterator: all checks passed; values =", list(Stream([1, 2, 3])))

try:
    check_manual(ManualDataStream)
except NotImplementedError as exc:
    print("TODO:", exc)


TODO: Exercise 1: __next__


In [15]:
stream = ManualDataStream([10, 20, 30])
iterator = iter(stream)
print("iter(stream) is stream:", iter(stream) is stream)
print("iter(iterator) is iterator:", iter(iterator) is iterator)
assert iter(stream) is not stream
assert iter(iterator) is iterator


iter(stream) is stream: False
iter(iterator) is iterator: True


**Explain:** `DataStream` describes the source; `DataStreamIterator` owns a cursor.
The stream is iterable. The cursor is both iterator and iterable. Tests above are
acceptance criteria, not a completed implementation. Solutions stay at the end.


## 7. Replacing the Iterator with a Generator
A generator function creates an iterator object; its suspended local state
replaces the explicit cursor. This is a new implementation stage, not the manual
exercise solution. **Predict:** Which iterator methods does the generator already have?


In [16]:
class DataStream:
    def __init__(self, data):
        self.data = data

    def __iter__(self):
        for item in self.data:
            yield item

GeneratorDataStream = DataStream
stream = DataStream([10, 20, 30])
g = iter(stream)
print("generator is Iterator:", isinstance(g, Iterator))
print("iter(g) is g:", iter(g) is g)
print("values:", list(g))
print("second traversal:", list(stream))
assert isinstance(g, Iterator) and iter(g) is g
assert list(stream) == [10, 20, 30]
assert list(g) == []
assert isinstance(stream, Iterable) and not isinstance(stream, Iterator)


generator is Iterator: True
iter(g) is g: True
values: [10, 20, 30]
second traversal: [10, 20, 30]


**Predict:** Creating a generator starts its body immediately — true or false?
Where does execution resume after each `yield`?


In [17]:
def count():
    print("start")
    yield 1
    print("middle")
    yield 2
    print("end")

g = count()
print("created; no body output yet")


created; no body output yet


In [18]:
print("first:", next(g))
print("second:", next(g))
try:
    next(g)
except StopIteration:
    print("finished")


start
first: 1
middle
second: 2
end
finished


### OPTIONAL / IF TIME — Inspect generator states
No frame internals needed. Compare the state before, between, and after requests.


In [19]:
from inspect import getgeneratorstate

state_demo = count()
print(getgeneratorstate(state_demo))
next(state_demo)
print(getgeneratorstate(state_demo))
list(state_demo)
print(getgeneratorstate(state_demo))


GEN_CREATED
start
GEN_SUSPENDED
middle
end
GEN_CLOSED


## 8. Lazy Processing
**Predict:** How many squares exist immediately after the next cell?
Do not turn the million-item stream into a list.


In [20]:
stream = DataStream(range(1_000_000))
squares = (x * x for x in stream)
print("pipeline created")
print("first square:", next(squares))
print("next three:", list(islice(squares, 3)))
assert next(squares) == 16


pipeline created
first square: 0
next three: [1, 4, 9]


```text
source → filter → transform → consumer
```
**Predict:** Exactly which source values must be requested to produce two even
squares? Inspect the trace, not just the final values.


In [21]:
requested = []
def observed_numbers():
    for value in range(1, 10):
        requested.append(value)
        print("read", value)
        yield value

stream = DataStream(observed_numbers())
result = (x * x for x in stream if x % 2 == 0)
assert requested == []
print("before consumption:", requested)
print("first result:", next(result))
assert requested == [1, 2]
print("second result:", next(result))
assert requested == [1, 2, 3, 4]


before consumption: []
read 1
read 2
first result: 4
read 3
read 4
second result: 16


Lazy pipelines delay work and can reduce peak memory or process infinite sources.
A consumer such as `list(...)` materializes all remaining results; an infinite
pipeline needs a bounded consumer such as `islice`.

A generator expression obtains its outermost iterator at construction, but delays
filtering and transformation. Here `DataStream.__iter__` is a generator, so even its
body is deferred. Laziness does **not** promise repeatability.


In [22]:
repeatable = DataStream([1, 2, 3])
one_shot = DataStream(iter([1, 2, 3]))
print("list source:", list(repeatable), list(repeatable))
print("iterator source:", list(one_shot), list(one_shot))
assert list(one_shot) == []


list source: [1, 2, 3] [1, 2, 3]
iterator source: [1, 2, 3] []


**Explain:** A fresh wrapper generator cannot rewind its exhausted source.
Which object owns the traversal state in the one-shot example?


## 9. Decorators
A decorator is an ordinary callable transformation: **callable → callable**.
Start without `@`. **Predict:** Which name will the wrapped function expose?


In [23]:
def log_calls_without_wraps(func):
    def wrapper(*args, **kwargs):
        print("Calling", func.__name__)
        return func(*args, **kwargs)
    return wrapper

def process_record(x):
    """Double one record."""
    return x * 2

original_process_record = process_record
process_record = log_calls_without_wraps(process_record)
print("value:", process_record(3))
print("name:", process_record.__name__)
print("doc:", process_record.__doc__)


Calling process_record
value: 6
name: wrapper
doc: None


The wrapper hid the useful metadata. Use `functools.wraps` for wrapper decorators.
It copies metadata and exposes `__wrapped__`; it does not validate behavior.


In [24]:
from functools import wraps

def log_calls(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print("Calling", func.__name__)
        return func(*args, **kwargs)
    return wrapper

process_record = log_calls(original_process_record)
print(process_record.__name__, "—", process_record.__doc__)
assert process_record.__name__ == "process_record"
assert process_record.__doc__ == "Double one record."
assert process_record.__wrapped__ is original_process_record
assert process_record(x=3) == 6


process_record — Double one record.
Calling process_record


Now use `@log_calls`: for this single decorator it means
`process_record = log_calls(process_record)` immediately after the definition.
**Predict:** Will creating the following pipeline log any calls?


In [25]:
@log_calls
def process_record(x):
    """Double one record."""
    return x * 2

processed = (process_record(x) for x in DataStream([1, 2, 3]))
print("created")


created


In [26]:
print("first:", next(processed))
print("remaining:", list(processed))
assert process_record.__name__ == "process_record"


Calling process_record
first: 2
Calling process_record
Calling process_record
remaining: [4, 6]


**Explain:** The generator delays the function calls; the decorator changes what
each call does. Decorators themselves are not a special-method protocol.
Parameterized decorators are outside the core lab; focus on this pipeline.


## 10. Exceptions in the Pipeline
**Predict:** Does invalid input fail when the pipeline is created or consumed?
Log a failure, then let its caller decide what to do.


In [27]:
def log_errors(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        try:
            return func(*args, **kwargs)
        except Exception as exc:
            print("Error:", type(exc).__name__, str(exc))
            raise
    return wrapper

@log_errors
def parse_record(text):
    """Parse one integer record."""
    return int(text)

parsed = (parse_record(text) for text in DataStream(["10", "abc", "30"]))
print("created without parsing")
print("first:", next(parsed))


created without parsing
first: 10


In [28]:
try:
    next(parsed)
except ValueError as exc:
    print("consumer caught:", type(exc).__name__)
else:
    raise AssertionError("ValueError was swallowed")
print("remaining after generator failure:", list(parsed))
assert list(parsed) == []
assert parse_record.__name__ == "parse_record"
assert parse_record.__doc__ == "Parse one integer record."


Error: ValueError invalid literal for int() with base 10: 'abc'
consumer caught: ValueError
remaining after generator failure: []


The decorator logs and re-raises; it does not return a fake successful value.
A bare `raise` preserves the current exception traceback; `raise exc` adds the
explicit re-raise location. An exception escaping a generator closes that generator:
`"30"` is not processed by this pipeline. Logging a generator **function call**
would not catch failures later during iteration; here we decorate the per-record parser.


## 11. Context Managers
### Exercise 2 — Context Manager
Extend the generator version of `DataStream` with resource lifetime tracking.
The resource is simulated: `opened` is a flag, no file is opened.
Iteration remains usable independently; `with` controls the lifetime flag.

Implement `__enter__` and `__exit__`: set `opened` to True/False, return the stream
on entry, and do not suppress exceptions. Print `open` and `close` to expose order.


```text
with stream as s:
    body

entry: __enter__() → s
body succeeds: __exit__(None, None, None)
body raises:   __exit__(exception type, exception value, traceback)
```

Cleanup runs after normal or exceptional body exit once entry succeeded.
If `__enter__` itself raises, Python does not call that manager's `__exit__`.
A truthy `__exit__` result suppresses a body exception; `False` or `None` lets it
propagate. Do not explicitly re-raise the received exception in `__exit__`.


In [29]:
class DataStream:
    def __init__(self, data):
        self.data = data
        self.opened = False

    def __iter__(self):
        for item in self.data:
            yield item

    def __enter__(self):
        # TODO: mark opened, print, and return this stream.
        raise NotImplementedError("Exercise 2: __enter__")

    def __exit__(self, exc_type, exc_value, traceback):
        # TODO: always clean up; do not suppress the exception.
        raise NotImplementedError("Exercise 2: __exit__")

ContextExerciseStream = DataStream


In [30]:
def check_context(Stream):
    resource = Stream([1, 2, 3])
    assert resource.opened is False
    with resource as entered:
        assert entered is resource
        assert resource.opened is True
        assert list(entered) == [1, 2, 3]
    assert resource.opened is False
    error = ValueError("broken")
    try:
        with resource:
            assert resource.opened is True
            raise error
    except ValueError as caught:
        assert caught is error
        print("body exception propagated")
    else:
        raise AssertionError("__exit__ suppressed the body exception")
    assert resource.opened is False
    print("Context manager: normal and exceptional cleanup passed")

try:
    check_context(ContextExerciseStream)
except NotImplementedError as exc:
    print("TODO:", exc)


TODO: Exercise 2: __enter__


**Predict → inspect → explain:** After implementing the methods, run the checks.
You should see `open`, `close`, then another `open`, `close`, followed by the
propagation message. Where would `return True` change control flow?
Temporarily try it: the check must fail. Restore non-suppressing behavior afterward.


## 12. Python 3.15: frozendict
**New in Python 3.15.** A frozen dictionary is an immutable mapping.
It is not a `dict` subclass. Hashing is possible when every key and value is hashable;
immutability of the mapping does not freeze nested objects.

Depend on `Mapping` when a function only reads configuration. Both `dict` and
`frozendict` can satisfy that interface.
API checked against the [Python 3.15 frozen dictionaries reference](https://docs.python.org/3.15/builtins/stdtypes.html#frozendict)
and [release notes](https://docs.python.org/3.15/whatsnew/3.15.html#pep-814-add-frozendict-built-in-type).


In [31]:
def endpoint(config: Mapping) -> str:
    return f"{config['host']}:{config['port']}"

print(endpoint({"host": "localhost", "port": 8000}))
if hasattr(builtins, "frozendict"):
    config = builtins.frozendict({"host": "localhost", "port": 8000})
    print("Mapping:", isinstance(config, Mapping))
    print("endpoint:", endpoint(config))
    assert isinstance(config, Mapping)
    assert endpoint(config) == "localhost:8000"
    try:
        config["port"] = 9000
    except TypeError as exc:
        print("immutable:", exc)
    else:
        raise AssertionError("frozendict unexpectedly allowed assignment")
    print("usable as a key:", {config: "development"}[config])
else:
    print("frozendict requires Python 3.15+")


localhost:8000
frozendict requires Python 3.15+


**Python 3.15+ syntax** (the executable cell above feature-checks it):
```python
config = frozendict({"host": "localhost", "port": 8000})
```
**Explain:** Which interface does `endpoint` need? Why would requiring `dict`
exclude a valid input? We use no third-party replacement on older interpreters.


## 13. Python 3.15: sentinel
**Predict:** If `get_value(default=None)` returns `None`, did the caller omit the
argument or explicitly supply a valid `None`? A unique marker separates the cases.
Traditional pattern, available on older Python:


In [32]:
MISSING = object()

def get_value(default=MISSING):
    if default is MISSING:
        return "default was omitted"
    return default

print(get_value())
print(get_value(None))
assert get_value() == "default was omitted"
assert get_value(None) is None
assert get_value(0) == 0


default was omitted
None


**New in Python 3.15:** `sentinel(name, /, *, repr=None)` creates a unique marker
with a useful representation. Compare markers using **`is`**, not `==`.

**Python 3.15+ syntax:**
```python
MISSING = sentinel("MISSING")
```
API checked against the [Python 3.15 sentinel reference](https://docs.python.org/3.15/builtins/functions.html#sentinel).


In [33]:
if hasattr(builtins, "sentinel"):
    MISSING = builtins.sentinel("MISSING")
    print("built-in marker:", MISSING)
    assert builtins.sentinel("MISSING") is not MISSING
else:
    MISSING = object()
    print("sentinel requires Python 3.15+; using object() fallback")

# Redefine: defaults are captured when a function is defined.
def get_value(default=MISSING):
    if default is MISSING:
        return "default was omitted"
    return default

print("omitted:", get_value())
print("explicit None:", get_value(None))
assert get_value() == "default was omitted"
assert get_value(None) is None
assert get_value(False) is False


sentinel requires Python 3.15+; using object() fallback
omitted: default was omitted
explicit None: None


**Explain:** Identity asks “is this the marker object?” Equality asks whether two
objects compare equal. User-defined equality can behave unexpectedly, so identity
is the right question for this contract. Why was `get_value` redefined after
replacing `MISSING`?


## 14. Final Integration Task
### Exercise 3 — Build the Whole Pipeline
Complete a fresh `DataStream`: lazy iteration, `opened` tracking, entry returning
self, and cleanup on every body exit without suppression. Complete `transform`
using the existing `@log_calls` decorator.

Use an iterable source without eagerly copying or consuming it. With list input,
independent traversals should work. With iterator input, do not promise rewind.
Expected result: **`[4, 8]`**, plus two `Calling transform` lines.
Consume inside `with`, so a resource-backed source would still be available.


In [34]:
class DataStream:
    def __init__(self, data):
        self.data = data
        self.opened = False

    def __iter__(self):
        # TODO: yield source values on demand.
        raise NotImplementedError("Exercise 3: __iter__")

    def __enter__(self):
        # TODO: begin the resource lifetime and return self.
        raise NotImplementedError("Exercise 3: __enter__")

    def __exit__(self, exc_type, exc_value, traceback):
        # TODO: clean up without suppressing errors.
        raise NotImplementedError("Exercise 3: __exit__")

@log_calls
def transform(x):
    """Double a selected record."""
    # TODO: transform one value.
    raise NotImplementedError("Exercise 3: transform")

IntegrationExerciseStream = DataStream


In [35]:
try:
    with DataStream([1, 2, 3, 4]) as stream:
        result = (transform(x) for x in stream if x % 2 == 0)
        values = list(result)
        print(values)
        assert values == [4, 8]
    assert stream.opened is False
except NotImplementedError as exc:
    print("TODO:", exc)


TODO: Exercise 3: __enter__


In [36]:
def check_integration(Stream, operation):
    check_context(Stream)
    seen = []
    def source():
        for value in (1, 2, 3, 4):
            seen.append(value)
            yield value

    stream = Stream(source())
    assert seen == []  # Construction must not consume the source.
    assert isinstance(stream, Iterable)
    assert Iterable not in Stream.__mro__
    with stream as entered:
        it = iter(entered)
        assert isinstance(it, Iterator)
        assert iter(it) is it
        assert seen == []  # Requesting an iterator must not consume records.
        result = (operation(x) for x in it if x % 2 == 0)
        assert seen == []
        assert next(result) == 4
        assert seen == [1, 2]
        assert list(result) == [8]
        assert seen == [1, 2, 3, 4]
    assert stream.opened is False
    assert list(Stream([])) == []
    repeatable = Stream([1, 2, 3])
    assert list(repeatable) == list(repeatable) == [1, 2, 3]
    assert operation.__name__ == "transform"
    assert operation.__doc__ == "Double a selected record."
    assert hasattr(operation, "__wrapped__")
    print("Integration: lazy traversal, protocol recognition, cleanup and metadata passed")

try:
    check_integration(IntegrationExerciseStream, transform)
except NotImplementedError as exc:
    print("TODO:", exc)


TODO: Exercise 3: __enter__


### OPTIONAL / IF TIME — Exercise 4: SupportsReset
Add `reset(self) -> None` to your integrated `DataStream`, then check the interface using AbstractBaseClass OR Protocol. Give reset a precise meaning: it sets a `processed` counter
to zero; increment the counter for each yielded item. It **does not rewind** the
underlying source or an existing iterator. No inheritance from SupportsReset.
The iterator implementation and two context methods remain as in Exercise 3.


In [ ]:
# choose whether you will use ABC or Protocol:
use_abc = True

if use_abc:
    class SupportsReset(ABC):
        @abstractmethod
        def reset(self) -> None:
            pass
else:
    @runtime_checkable
    class SupportsReset(Protocol):
        def reset(self) -> None:
            ...

def check_reset(Stream):
    stream = Stream([10, 20, 30])
    assert isinstance(stream, SupportsReset)
    assert SupportsReset not in Stream.__mro__
    assert stream.processed == 0
    it = iter(stream)
    assert next(it) == 10 and stream.processed == 1
    stream.reset()
    assert stream.processed == 0
    assert next(it) == 20 and stream.processed == 1
    print("SupportsReset recognized; absent from MRO; counter reset without rewind")

# After adding processed and reset to your class, run:
# check_reset(DataStream)
print("Optional task: add counter/reset to your completed DataStream")


Optional task: add counter/reset to your completed DataStream


## 15. Summary
| Protocol / abstraction | Mechanism | What to remember |
|---|---|---|
| Iterable | `__iter__`; ABC / `__subclasshook__` recognition | Virtual compatibility need not appear in MRO |
| Iterator | `__iter__`, `__next__` | Holds traversal state; every iterator is iterable |
| Generator | Iterator created by a generator function | Suspends at yield; convenient iterator implementation |
| Context manager | `__enter__`, `__exit__` | Cleanup on body exit; truthy exit suppresses exceptions |
| `typing.Protocol` | Static structural typing | Runtime checks require `@runtime_checkable` and remain shallow |
| Decorator | callable → callable | `@` rebinds a name; wrappers should preserve metadata |
| `frozendict` | Immutable Mapping | Read-only mapping interface; Python 3.15 feature check |
| `sentinel` | Unique identity marker | Distinguish omission from None with `is` |

**Exit questions:**
1. Why is `isinstance(MyCollection(), Iterable)` true without inheritance?
2. Why can `OldSequence` be iterated despite a false Iterable check?
3. Which object stores traversal state in each DataStream implementation?
4. When does the lazy parser raise, and who handles its error?
5. What ensures `opened` becomes False after an exception?

Python often asks **“What can this object do?”** rather than
**“What exact class is this object?”** Syntax requests a behavior; special methods
implement it; ABCs and type protocols provide different ways to describe or recognize it.
